# A4. 差评批量分析 Notebook

> **配套模块**: [A4 客服与售后](../paths/a-operators/a4-customer-service.md)
>
> **功能**: 上传差评 CSV → 自动分类 + 频率统计 + 改善建议 + 可视化
>
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kangise/ecommerce-ai-skills/blob/main/notebooks/a4-negative-review-analysis.ipynb)

---

## 1. 安装依赖

In [ ]:
!pip install -q pandas numpy plotly wordcloud matplotlib

## 2. 加载差评数据

In [ ]:
import pandas as pd
import re
from collections import Counter
from pathlib import Path

REVIEW_CSV = Path('negative-reviews.csv')
if not REVIEW_CSV.is_file():
    raise FileNotFoundError('请上传真实差评 CSV 并命名为 negative-reviews.csv')

df = pd.read_csv(REVIEW_CSV)
required = {'rating', 'title', 'body', 'date', 'helpful_votes'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'差评 CSV 缺少必需列: {sorted(missing)}')
if df.empty:
    raise ValueError('差评 CSV 不能为空')
df['rating'] = pd.to_numeric(df['rating'], errors='raise')
df['helpful_votes'] = pd.to_numeric(df['helpful_votes'], errors='raise')
if not df['rating'].between(1, 2).all():
    raise ValueError('本分析仅接受 1–2 星差评')
if (df['helpful_votes'] < 0).any():
    raise ValueError('helpful_votes 不能为负数')
df['date'] = pd.to_datetime(df['date'], errors='raise')
df['title'] = df['title'].fillna('').astype(str).str.strip()
df['body'] = df['body'].fillna('').astype(str).str.strip()
if ((df['title'] == '') & (df['body'] == '')).any():
    raise ValueError('每条差评必须包含 title 或 body')
df['full_text'] = (df['title'] + '. ' + df['body']).str.strip()
print(f'Loaded {len(df)} negative reviews (1-2 stars)')
print(f'Rating distribution: {df["rating"].value_counts().to_dict()}')
df.head()

## 3. 关键词提取与问题分类

In [ ]:
# 定义问题类别和关键词
ISSUE_CATEGORIES = {
    '🔋 电池续航': ['battery', 'charge', 'dies', 'hours', 'drain', 'last'],
    '📶 蓝牙连接': ['bluetooth', 'connection', 'disconnect', 'drops', 'pair', 'connect'],
    '🔨 做工质量': ['broke', 'broken', 'cheap', 'plastic', 'build', 'quality', 'fell apart'],
    '🔊 音质问题': ['sound', 'audio', 'bass', 'tinny', 'volume', 'louder', 'delay'],
    '👂 佩戴舒适': ['fit', 'comfortable', 'ear', 'pain', 'fall', 'size', 'tip'],
    '🔇 降噪效果': ['noise', 'cancellation', 'anc', 'cancel'],
    '📦 充电盒': ['case', 'charging case', 'lid', 'led', 'indicator'],
    '🛠️ 功能故障': ['stopped working', 'defective', 'malfunction', 'dead']
}

def classify_issue(text):
    text_lower = text.lower()
    matches = []
    for category, keywords in ISSUE_CATEGORIES.items():
        if any(kw in text_lower for kw in keywords):
            matches.append(category)
    return matches if matches else ['❓ 其他']

df['issues'] = df['full_text'].apply(classify_issue)
df['primary_issue'] = df['issues'].apply(lambda x: x[0])

# 统计
issue_counts = df.explode('issues')['issues'].value_counts()
print('=== 差评问题分类排名 ===')
for issue, count in issue_counts.items():
    pct = count / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f'{issue}: {count} ({pct:.0f}%) {bar}')

## 4. 可视化

In [ ]:
import plotly.express as px
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# 问题分布饼图
fig = px.pie(values=issue_counts.values, names=issue_counts.index,
             title='差评问题分布', hole=0.3)
fig.show()

# 问题趋势（月度）
df['month'] = pd.to_datetime(df['date']).dt.to_period('M').astype(str)
exploded = df.explode('issues')
monthly_issues = exploded.groupby(['month', 'issues']).size().reset_index(name='count')
fig = px.line(monthly_issues, x='month', y='count', color='issues',
              title='差评问题月度趋势（哪些问题在恶化？）')
fig.show()

# 词云
all_text = ' '.join(df['full_text'].tolist())
wc = WordCloud(width=800, height=400, background_color='white', max_words=60, colormap='Reds').generate(all_text)
plt.figure(figsize=(12, 6))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('差评关键词词云', fontsize=16)
plt.show()

## 5. 生成改善建议

In [ ]:
print('=== 产品改善优先级建议 ===')
print()
for i, (issue, count) in enumerate(issue_counts.head(5).items(), 1):
    pct = count / len(df) * 100
    severity = '🔴 紧急' if pct > 20 else ('🟡 重要' if pct > 10 else '🟢 关注')
    print(f'{i}. {issue} — {count} 条差评 ({pct:.0f}%) [{severity}]')
    # 显示该类别的典型差评
    examples = df[df['primary_issue'] == issue]['full_text'].head(2)
    for ex in examples:
        print(f'   → "{ex[:80]}..."')
    print()

print('\n=== 行动建议 ===')
print('P0 (本周): 解决排名第 1 的问题 — 直接影响退货率和评分')
print('P1 (本月): 解决排名第 2-3 的问题 — 在 Listing 中管理预期')
print('P2 (下月): 在 Q&A 中预埋排名前 5 问题的回答（Rufus 优化）')

## 6. 导出

In [ ]:
# 导出带分类的差评数据
df[['date', 'rating', 'title', 'body', 'primary_issue', 'helpful_votes']].to_csv('negative_reviews_classified.csv', index=False)

# 导出问题摘要
summary = issue_counts.reset_index()
summary.columns = ['Issue', 'Count']
summary['Percentage'] = (summary['Count'] / len(df) * 100).round(1)
summary.to_csv('issue_summary.csv', index=False)

print('Exported: negative_reviews_classified.csv, issue_summary.csv')